## Setup and Predictions

In [ ]:
import pandas as pd
import pickle
import numpy as np

# Load original text to see the actual messages
df_raw = pd.read_csv('../data/raw/spam.tsv', sep='\t', header=None, names=['label', 'message'])
df_raw['label_num'] = df_raw['label'].map({'ham': 0, 'spam': 1})

# Load processed text for prediction
df_clean = pd.read_csv('../data/processed/cleaned_spam.csv')
df = pd.merge(df_raw, df_clean, left_index=True, right_index=True, how='inner')

with open('../models/bow_vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)
with open('../models/naive_bayes.pkl', 'rb') as f:
    model = pickle.load(f)

# Predict on entire dataset for error analysis
X_bow = vectorizer.transform(df['clean_message'].fillna(''))
df['prediction'] = model.predict(X_bow)
df['confidence'] = np.max(model.predict_proba(X_bow), axis=1)

## False Positives

In [ ]:
# False Positives: The most dangerous error in spam filtering
false_positives = df[(df['label_num'] == 0) & (df['prediction'] == 1)]

print(f"Total False Positives: {len(false_positives)}")
print("\nExamples of False Positives:")
for text in false_positives.sort_values(by='confidence', ascending=False)['message'].head(10):
    print(f"- {text}")
    print("---")

## False Negatives

In [ ]:
# False Negatives: Annoying, but less destructive
false_negatives = df[(df['label_num'] == 1) & (df['prediction'] == 0)]

print(f"Total False Negatives: {len(false_negatives)}")
print("\nExamples of False Negatives:")
for text in false_negatives.sort_values(by='confidence', ascending=False)['message'].head(10):
    print(f"- {text}")
    print("---")